In [340]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

Importing Dataset:

In [341]:
df = pd.read_csv("btc_daily_data.csv")
df['Date'] = pd.to_datetime(df['Date'])

Feature engineering

In [342]:
# Daily return
df['return'] = df['Close'].pct_change()

# Moving average
df['ma_7'] = df['Close'].rolling(window=7).mean()

# Volatility (std of returns)
df['volatility_7'] = df['return'].rolling(window=7).std()

# High-Low range
df['hl_range'] = df['High'] - df['Low']

# Target: next day up/down
df['target'] = (df['Close'].shift(-1) > df['Close']).astype(int)

# Drop NaN rows
df = df.dropna()

In [343]:
df.corr(numeric_only=True)['target'].sort_values(ascending=False)

target          1.000000
Volume          0.018803
volatility_7    0.006616
hl_range       -0.029685
return         -0.035247
Open           -0.038640
ma_7           -0.038956
Low            -0.039107
High           -0.039123
Close          -0.039779
Name: target, dtype: float64

In [344]:

df = df.dropna()

X = df.drop(columns=['target','Date'])
y = df['target']

In [345]:
X = X.replace([np.inf, -np.inf], np.nan)
X = X.dropna()
y = y.loc[X.index]

In [346]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

model = LogisticRegression()
model.fit(X_train, y_train)

print("Accuracy:", model.score(X_test, y_test))
print(y.value_counts(normalize=True))

Accuracy: 0.5014492753623189
target
1    0.531232
0    0.468768
Name: proportion, dtype: float64


In [347]:
baseline = y_test.value_counts(normalize=True).max()
print("Baseline:", baseline)

Baseline: 0.5101449275362319


In [348]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, model.predict(X_test)))

[[444  63]
 [453  75]]


In [354]:
import json

filename='export'
original_weights = model.coef_[0]
original_bias = model.intercept_[0]

mean = scaler.mean_
scale = scaler.scale_

# Transform weights
new_weights = original_weights / scale

# Transform bias
new_bias = original_bias - np.sum((original_weights * mean) / scale)

model_data = {
    "weights": new_weights.tolist(),
    "bias": float(new_bias),
}

with open(filename, "w") as f:
    json.dump(model_data, f, indent=4)

print("Transformed model exported (no scaling needed in JS)")

Transformed model exported (no scaling needed in JS)
